# Analisis Pemilihan Model (Library)
**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini melanjutkan notebook 02 dengan menggunakan **library scikit-learn**.
Membandingkan 4 algoritma klasifikasi: **Logistic Regression, Decision Tree,
Random Forest, dan SVM**. Plus eksperimen dengan scaling fitur dan penanganan
class imbalance (class_weight).

Dataset: Stroke Prediction (5.110 pasien, kolom numerik: age, hypertension,
heart_disease, avg_glucose_level, bmi). Target: stroke.

## 1. Muat Data dan Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

URL = "https://raw.githubusercontent.com/ravi-arnan/machine-learning/main/data/healthcare-stroke-data.csv"
df = pd.read_csv(URL)

df = df[["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]].copy()
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Ukuran dataset:", df.shape)
print()
print("5 baris pertama:")
df.head()

In [ ]:
X = df.drop(columns=["stroke"]).values
y = df["stroke"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} baris")
print(f"Test:  {X_test.shape[0]} baris")

# Scaling untuk model yang sensitif skala (Logistic Regression, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nProporsi stroke:")
print(f"  Train: {y_train.mean():.3f}")
print(f"  Test:  {y_test.mean():.3f}")

## 2. Fungsi Uji Model

Untuk menghindari kode berulang.

In [ ]:
def evaluasi_model(model, X_test, y_test):
    """Prediksi dan cetak metrik."""
    y_pred = model.predict(X_test)

    hasil = {
        "akurasi": round(accuracy_score(y_test, y_pred), 4),
        "presisi": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
    }
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    hasil["tp"] = tp
    hasil["tn"] = tn
    hasil["fp"] = fp
    hasil["fn"] = fn
    return hasil, y_pred


def cetak_hasil(nama, hasil):
    print(f"\n{'=' * 50}")
    print(f"  {nama}")
    print(f"{'=' * 50}")
    print(f"  Akurasi:  {hasil['akurasi']}")
    print(f"  Presisi:  {hasil['presisi']}")
    print(f"  Recall:   {hasil['recall']}")
    print(f"  F1-score: {hasil['f1']}")
    print(f"  TP={hasil['tp']}  FP={hasil['fp']}  FN={hasil['fn']}  TN={hasil['tn']}")

## 3. Logistic Regression

In [ ]:
# Tanpa scaling vs dengan scaling
for nama, X_tr, X_te in [
    ("Logistic Regression (tanpa scaling)", X_train, X_test),
    ("Logistic Regression (dengan scaling)", X_train_scaled, X_test_scaled),
]:
    for imbang in [None, "balanced"]:
        label = f"{nama}, class_weight={imbang}"
        model = LogisticRegression(max_iter=5000, class_weight=imbang)
        model.fit(X_tr, y_train)
        hasil, _ = evaluasi_model(model, X_te, y_test)
        cetak_hasil(label, hasil)

## 4. Decision Tree

In [ ]:
for kedalaman in [3, 5, 10, None]:
    for imbang in [None, "balanced"]:
        label = f"Decision Tree (max_depth={kedalaman}, class_weight={imbang})"
        model = DecisionTreeClassifier(max_depth=kedalaman, class_weight=imbang, random_state=42)
        model.fit(X_train, y_train)
        hasil, _ = evaluasi_model(model, X_test, y_test)
        cetak_hasil(label, hasil)

## 5. Random Forest

In [ ]:
for n in [50, 100, 200]:
    for imbang in [None, "balanced"]:
        label = f"Random Forest (n={n}, class_weight={imbang})"
        model = RandomForestClassifier(n_estimators=n, class_weight=imbang, random_state=42)
        model.fit(X_train, y_train)
        hasil, _ = evaluasi_model(model, X_test, y_test)
        cetak_hasil(label, hasil)

## 6. SVM (Support Vector Machine)

In [ ]:
for kernel in ["linear", "rbf"]:
    for imbang in [None, "balanced"]:
        label = f"SVM ({kernel}, class_weight={imbang})"
        model = SVC(kernel=kernel, class_weight=imbang, random_state=42)
        model.fit(X_train_scaled, y_train)
        hasil, _ = evaluasi_model(model, X_test_scaled, y_test)
        cetak_hasil(label, hasil)

## 7. Perbandingan Semua Model (F1-score)

Rangkuman model terbaik tiap keluarga.

In [ ]:
model_terbaik = [
    ("Logistic Regression + scaling + balanced",
     LogisticRegression(max_iter=5000, class_weight="balanced").fit(X_train_scaled, y_train),
     X_test_scaled),
    ("Decision Tree (max_depth=5, balanced)",
     DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_train, y_train),
     X_test),
    ("Random Forest (n=100, balanced)",
     RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42).fit(X_train, y_train),
     X_test),
    ("SVM (rbf, balanced)",
     SVC(kernel="rbf", class_weight="balanced", random_state=42).fit(X_train_scaled, y_train),
     X_test_scaled),
]

print(f"{'Model':<45} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 77)

for nama, model, X_te in model_terbaik:
    hasil, _ = evaluasi_model(model, X_te, y_test)
    print(f"{nama:<45} {hasil['akurasi']:>8} {hasil['presisi']:>8} {hasil['recall']:>8} {hasil['f1']:>8}")

## 8. Model Paling Recommended

Pilih model dengan F1 tertinggi karena class imbalance membuat akurasi menyesatkan.

In [ ]:
print("Kesimpulan:")
print()
print("Karena dataset tidak seimbang (stroke hanya 5%), metrik utama adalah F1-score,")
print("bukan akurasi. Model terbaik adalah yang punya F1 tertinggi.")
print()
print("Catatan: Random Forest dan SVM dengan kernel rbf cenderung lebih baik")
print("untuk dataset dengan hubungan non-linear dan class imbalance.")
print()
print("Cross-validation dan hyperparameter tuning lebih lanjut bisa dilakukan")